# Process Model Quality Metrics
Task 3.3.2: all relevant PM4Py quality metrics (fitness, precision, generalization,
simplicity) for every discovered process model.

The Petri nets are not rediscovered here; they are loaded from `reports/initial_discovery_algorithms/petri_net_<miner>.pnml`
(written by `python -m src.miners`). Every metric is implemented as its own function in
`src/metrics.py` and called in its own cell, so the slow ones (alignments) can be run separately.
To compute everything in one go, with the results saved after each metric, use
`python -m src.metrics` (or `metrics.run_all()`). Results are collected in `results` and summarised at the end.

## 1. Imports and Settings

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import pm4py

from src import metrics
from src.config import INITIAL_DISCOVERY
from src.miners import load_petri_net_pnml

pd.set_option("display.max_columns", None)

## 2. Load Data and Models
Same log as in `src/miners.py`: complete cases, one event per started work item.

In [ ]:
log = metrics.load_log()

In [ ]:
models = {name: load_petri_net_pnml(name) for name in ["inductive", "heuristics"]}
results = {name: {} for name in models}

## 3. Inductive Miner (IMf)

In [ ]:
model = models["inductive"]
res = results["inductive"]
res.update(metrics.size(model))

### 3.1 Soundness

In [ ]:
# the inductive miner returns sound (block-structured) nets by construction, so the slow soundness check is skipped
res["sound"] = True
res["sound"]

### 3.2 Simplicity

In [ ]:
res.update(metrics.simplicity_arc_degree(model))
res["simplicity_arc_degree"]

In [ ]:
res.update(metrics.simplicity_extended_cardoso(model))
res["simplicity_extended_cardoso"]

In [ ]:
# builds the full reachability graph, which is only feasible for sound nets,
if res["sound"]:
    res.update(metrics.simplicity_extended_cyclomatic(model))
else:
    res["simplicity_extended_cyclomatic"] = float("nan")
res["simplicity_extended_cyclomatic"]

In [ ]:
res.update(metrics.simplicity_density(model))
res["simplicity_density"]

In [ ]:
res.update(metrics.node_count(model))
res["node_count"]

### 3.3 Fitness

In [ ]:
res.update(metrics.fitness_tbr(model, log))
{k: res[k] for k in ["fitness_tbr", "fit_traces_tbr"]}

In [ ]:
res.update(metrics.fitness_alignments(model, log))
{k: res[k] for k in ["fitness_align", "fit_traces_align"]}

### 3.4 Precision

In [ ]:
res.update(metrics.precision_tbr(model, log))
res["precision_tbr"]

In [ ]:
res.update(metrics.precision_alignments(model, log))
res["precision_align"]

### 3.5 Generalization

In [ ]:
res.update(metrics.generalization_tbr(model, log))
res["generalization_tbr"]

## 4. Heuristics Miner

In [ ]:
model = models["heuristics"]
res = results["heuristics"]
res.update(metrics.size(model))
pm4py.view_petri_net(*model)

### 4.1 Soundness

In [ ]:
res.update(metrics.soundness(model))
res["sound"]

### 4.2 Simplicity

In [ ]:
res.update(metrics.simplicity_arc_degree(model))
res["simplicity_arc_degree"]

In [ ]:
res.update(metrics.simplicity_extended_cardoso(model))
res["simplicity_extended_cardoso"]

In [ ]:
# builds the full reachability graph, which is only feasible for sound nets,
if res["sound"]:
    res.update(metrics.simplicity_extended_cyclomatic(model))
else:
    res["simplicity_extended_cyclomatic"] = float("nan")
res["simplicity_extended_cyclomatic"]

In [ ]:
res.update(metrics.simplicity_density(model))
res["simplicity_density"]

In [ ]:
res.update(metrics.node_count(model))
res["node_count"]

### 4.3 Fitness

In [ ]:
res.update(metrics.fitness_tbr(model, log))
{k: res[k] for k in ["fitness_tbr", "fit_traces_tbr"]}

In [ ]:
res.update(metrics.fitness_alignments(model, log))
{k: res[k] for k in ["fitness_align", "fit_traces_align"]}

### 4.4 Precision

In [ ]:
res.update(metrics.precision_tbr(model, log))
res["precision_tbr"]

In [ ]:
res.update(metrics.precision_alignments(model, log))
res["precision_align"]

### 4.5 Generalization

In [ ]:
res.update(metrics.generalization_tbr(model, log))
res["generalization_tbr"]

## 5. Summary

In [ ]:
for res in results.values():
    for kind in ["tbr", "align"]:
        if f"fitness_{kind}" in res and f"precision_{kind}" in res:
            res[f"f_score_{kind}"] = metrics.f_score(res[f"fitness_{kind}"], res[f"precision_{kind}"])

summary = pd.DataFrame(results)
summary.to_csv(INITIAL_DISCOVERY / "metrics.csv")
summary